# Wafer Map TDA

Synthetic wafer maps → persistent-homology features → defect classifier → embeddings and a Mapper graph.

Run the cells from top to bottom. A full run takes about a minute on a laptop-class CPU. The data is synthetic, so the scores below are not real-fab results (see `docs/REVIEW.md`).

In [ ]:
# Google Colab only: install the package. On a local machine, run `pip install -e '.[notebook]'`
# from the repository root instead (see the README).
import sys
if "google.colab" in sys.modules:
    %pip install -q "wafer-map-tda[notebook] @ git+https://github.com/janhavi-giri/wafer-map-tda.git"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from wafer_map_tda import SyntheticWaferMapGenerator, TDAFeatureExtractor, WaferDefectClassifier

## Research demonstration
Synthetic WM811K-style patterns, not measured WM811K data. See docs/REVIEW.md for limitations. This working copy fixes execution and evaluation issues but retains experimental feature choices.


## 1. Generate synthetic wafer maps

In [ ]:
np.random.seed(42)
generator = SyntheticWaferMapGenerator(wafer_size=26)
wafer_maps, labels = generator.generate_dataset(n_samples=1000)
dataset_path = generator.save_dataset_to_folders(wafer_maps, labels, base_path='./data', save_images=False)


In [ ]:
 # Visualize samples
generator.visualize_samples(wafer_maps, labels, n_samples=9)

## 2. Extract TDA features and train a classifier

The default demo uses H0 and H1 only (34 features). Set `max_dimension=2` for the original 47-feature definition; it is much slower.

In [ ]:
# H0/H1 demo; set max_dimension=2 to reproduce the original 47-feature definition.
tda_extractor = TDAFeatureExtractor(max_dimension=1)
classifier = WaferDefectClassifier(tda_extractor)
results = classifier.train(wafer_maps, labels, classifier_type='random_forest')
classifier.visualize_results(results)


## 3. Evaluate on a second synthetic dataset

A different random seed, but the same generator, so this checks repeatability within the simulator only.

In [ ]:
# Independent seed, same synthetic generator: not a real-fab validation set.
np.random.seed(43)
new_wafer_maps, new_labels = generator.generate_dataset(n_samples=1000)
predictions, probabilities = classifier.predict(new_wafer_maps)


In [ ]:
for i in range(min(10, len(predictions))):
    print(f'Sample {i+1}: Predicted: {predictions[i]}, True: {new_labels[i]}')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Evaluate the classifier on the new dataset
print("Evaluating classifier on the new dataset...")

# Generate classification report
report = classification_report(new_labels, predictions, output_dict=True)
print("\nClassification Report:")
report_df = pd.DataFrame(report).transpose()
display(report_df)

# Generate confusion matrix
cm = confusion_matrix(new_labels, predictions, labels=classifier.label_encoder.classes_)
print("\nConfusion Matrix:")
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classifier.label_encoder.classes_,
            yticklabels=classifier.label_encoder.classes_)
plt.title('Confusion Matrix for independent synthetic dataset')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

# Calculate overall accuracy
accuracy = np.sum(predictions == new_labels) / len(new_labels)
print(f"\nOverall Accuracy: {accuracy:.4f}")

## 4. Explore the feature space (held-out dataset)

In [ ]:
# Step 1: Feature Extraction
# Use the existing tda_extractor to extract features from the new_wafer_maps
extracted_features, feature_names = tda_extractor.fit_transform(new_wafer_maps)

print(f"Extracted features shape: {extracted_features.shape}")
print(f"Number of features: {len(feature_names)}")

In [ ]:
# UMAP embedding of the held-out TDA features, standardized with the training scaler
import umap

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
embedding = reducer.fit_transform(classifier.scaler.transform(extracted_features))
print(f"UMAP embedding shape: {embedding.shape}")

embedding_df = pd.DataFrame(embedding, columns=['UMAP_1', 'UMAP_2'])
embedding_df['Label'] = new_labels

plt.figure(figsize=(12, 8))
sns.scatterplot(data=embedding_df, x='UMAP_1', y='UMAP_2', hue='Label', legend='full', palette='viridis')
plt.title('UMAP Projection of TDA Features (held-out dataset)')
plt.xlabel('UMAP_1')
plt.ylabel('UMAP_2')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True)
plt.show()

In [ ]:
# t-SNE embedding of the same standardized held-out features
from sklearn.manifold import TSNE

extracted_features_scaled = classifier.scaler.transform(extracted_features)
tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(extracted_features) - 1))
tsne_results = tsne.fit_transform(extracted_features_scaled)
print(f"t-SNE results shape: {tsne_results.shape}")

tsne_df = pd.DataFrame(tsne_results, columns=['TSNE-Component-1', 'TSNE-Component-2'])
tsne_df['label'] = new_labels

plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='TSNE-Component-1', y='TSNE-Component-2', hue='label',
    palette=sns.color_palette('hsv', tsne_df['label'].nunique()),
    data=tsne_df, legend='full', alpha=0.6,
)
plt.title('t-SNE Visualization of TDA Features (held-out dataset)')
plt.xlabel('TSNE Component 1')
plt.ylabel('TSNE Component 2')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.decomposition import PCA
scaled_features = classifier.scaler.transform(extracted_features)
pca = PCA(n_components=2)
reduced_features = pca.fit_transform(scaled_features)
print('Explained variance:', pca.explained_variance_ratio_.sum())


In [ ]:
# Step 3: Clustering
from sklearn.cluster import KMeans

# Apply K-Means clustering
n_clusters = 9 # Number of clusters based on known defect types
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(reduced_features)

print(f"Cluster labels shape: {cluster_labels.shape}")
print(f"Unique cluster labels: {np.unique(cluster_labels)}")

In [ ]:
# Step 4: Visualization
plt.figure(figsize=(10, 8))
scatter = plt.scatter(reduced_features[:, 0], reduced_features[:, 1], c=cluster_labels, cmap='viridis', s=10)
plt.title('Clustering of TDA Features (PCA Reduced)')
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.colorbar(scatter, label='Cluster Label')
plt.grid(True)
plt.show()

In [ ]:
# Step 5: Evaluation (Optional)
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score

# Evaluate clustering using Silhouette Score (measures how similar a data point is to its own cluster compared to other clusters)
# A higher Silhouette Score indicates better-defined clusters.
silhouette_avg = silhouette_score(reduced_features, cluster_labels)
print(f"Silhouette Score: {silhouette_avg:.4f}")

# Evaluate clustering against true labels (since true labels are available)
# Use Adjusted Rand Index (ARI) and Normalized Mutual Information (NMI)
# These metrics measure the similarity between the clustering and the true labels, ignoring permutations.
# Values closer to 1 indicate better agreement.
true_labels_encoded = classifier.label_encoder.transform(new_labels) # Encode true labels to match cluster label format

ari_score = adjusted_rand_score(true_labels_encoded, cluster_labels)
nmi_score = normalized_mutual_info_score(true_labels_encoded, cluster_labels)

print(f"Adjusted Rand Index (ARI): {ari_score:.4f}")
print(f"Normalized Mutual Information (NMI): {nmi_score:.4f}")

## Interpret newly computed results
Historical scores belong to the original notebook. Clusters use nine known categories as a prior. Silhouette here is computed in the same PCA space as clustering. ARI/NMI describe agreement, not supervised accuracy.


## 5. Mapper graph

In [ ]:
# Mapper graph of the held-out TDA features
import kmapper
from kmapper import Cover
from sklearn.cluster import DBSCAN
from sklearn.decomposition import PCA

mapper = kmapper.KeplerMapper(verbose=0)
scaled_extracted_features = StandardScaler().fit_transform(extracted_features)

# Lens: the first two principal components of the standardized features
lens = mapper.project(scaled_extracted_features, projection=PCA(n_components=2), scaler=None)

# DBSCAN clusters the samples inside each cover region. A larger eps keeps more samples in the
# graph; samples labeled as noise in every region are left out. See
# wafer_tda_mapper_eps2_all_classes.ipynb for a run where every defect type appears.
graph = mapper.map(
    lens,
    scaled_extracted_features,
    clusterer=DBSCAN(eps=1.0, min_samples=3),
    cover=Cover(n_cubes=10, perc_overlap=0.5),
)

mapper.visualize(
    graph,
    path_html="tda_mapper_graph.html",
    title="TDA Mapper Graph of Wafer Map Features",
    custom_tooltips=np.array(new_labels),
)
print("Mapper graph generated and saved as tda_mapper_graph.html")

In [ ]:
from pathlib import Path
print(Path('tda_mapper_graph.html').resolve())
